In [ ]:
!pip install ninja
import os
import torch
from torch.utils.cpp_extension import load_inline

cuda_source = """
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <cmath>
#include <ATen/cuda/CUDAContext.h>

#ifndef M_PI
#define M_PI 3.14159265358979323846
#endif

const int BLOCK_SIZE = 256;

template <typename scalar_t>
__global__ void adamv_prepare_kernel(
    float* __restrict__ exp_avg,
    float* __restrict__ exp_avg_sq,
    float* __restrict__ direcao_buffer,
    const scalar_t* __restrict__ grad,
    float beta1, float beta2, float bias_correction2, float eps, int step, int numel) {
    
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < numel) {
        float g = static_cast<float>(grad[idx]);
        float m = exp_avg[idx];
        float v = exp_avg_sq[idx];

        v = beta2 * v + (1.0f - beta2) * g * g;
        exp_avg_sq[idx] = v;

        float v_hat = v / bias_correction2;
        float sqrt_v_hat = sqrt(v_hat);

        // BRCM: Excess Shock Isolator (delta)
        // Tighter tuning: 1.0x margin to prevent excess momentum in sharp minima
        float delta = fmaxf(0.0f, std::abs(g) - 1.5f * sqrt_v_hat);
        float denom_brcm = sqrt_v_hat + delta + eps;
        float bakh_residual = (delta * delta) / (2.0f * denom_brcm);
        
        // Full curvature shift to avoid carrying too much momentum
        float curvature_shift = bakh_residual / (sqrt_v_hat + eps);
        float beta1_dynamic = beta1 * std::exp(-0.5f * curvature_shift);
        
        m = beta1_dynamic * m + (1.0f - beta1_dynamic) * g;
        exp_avg[idx] = m;
        
        float bias_correction1_dynamic = 1.0f - std::pow(beta1_dynamic, static_cast<float>(step));

        float m_hat = m / bias_correction1_dynamic;
        
        direcao_buffer[idx] = m_hat / (sqrt_v_hat + eps);
    }
}

template <typename scalar_t>
__global__ void adamv_update_kernel(
    scalar_t* __restrict__ params,
    const scalar_t* __restrict__ grad,
    float* __restrict__ exp_avg,
    float* __restrict__ exp_avg_sq,
    const float* __restrict__ direcao_buffer,
    const float* __restrict__ norm_tensor_ptr,
    float progresso, float cooling_factor, float bakh_thresh_eff, float bias_correction2, float eps, 
    float wd_factor, float lr_max, float weight_decay, int numel, int D, int step,
    bool enable_cooling, bool enable_brake,
    bool omni_triggered, uint32_t punning_mask) {
    
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < numel) {
        scalar_t p = params[idx];
        
        float g = static_cast<float>(grad[idx]);
        float dir = direcao_buffer[idx];
        float v = exp_avg_sq[idx];
        
        float lr_efetivo = lr_max;
        if (enable_cooling) {
            float norm_dir = (*norm_tensor_ptr) / sqrt(static_cast<float>(D));
            float envelope = (1.0f + progresso) / (progresso + norm_dir + eps);
            lr_efetivo = lr_max * min(envelope * cooling_factor, 1.5f);
        }
        
        float a = lr_efetivo * dir;
        float v_hat = v / bias_correction2;
        float sqrt_v = sqrt(v_hat);
        
        float step_size = a;
        if (enable_brake) {
            bool explosao_mask = std::abs(g) > (bakh_thresh_eff * sqrt_v);
            if (explosao_mask) {
                // Dimensional Mismatch Fix: parameter absolute magnitude
                // Tightened Tuning: Stronger brake penalty for late-stage stability
                float denom = std::abs(static_cast<float>(p)) + (std::abs(a) * 2.0f) + eps;
                float correction = (a * a) / (2.0f * denom);
                float bakhshali_brake = a - copysignf(1.0f, a) * correction;
                step_size = bakhshali_brake;
            }
        }
        
        // Orthogonal Truncated Levy Flight Injection
        uint32_t seed = (static_cast<uint32_t>(step) * 31337) ^ static_cast<uint32_t>(idx);
        seed = (seed * 1664525) + 1013904223;
        float u = (static_cast<float>(seed) / 4294967296.0f) * 0.999f + 0.0005f;
        float cauchy = tanf(M_PI * (u - 0.5f));
        cauchy = fmaxf(-500.0f, fminf(500.0f, cauchy));
        float levy_scale = sqrt_v / (sqrt_v + 0.1f);
        float levy_mult = 0.01f * fmaxf(0.0f, 1.0f - progresso);
        float levy_jump = lr_efetivo * levy_mult * cauchy * levy_scale;
        
        step_size -= levy_jump;
        
        if (weight_decay != 0.0f) {
            p = static_cast<scalar_t>(static_cast<float>(p) * (1.0f - lr_max * weight_decay * wd_factor));
        }
        
        params[idx] = static_cast<scalar_t>(static_cast<float>(p) - step_size);
        
        if (omni_triggered) {
            if (sizeof(scalar_t) == 4) {
                float p_new = static_cast<float>(params[idx]);
                uint32_t p_int = __float_as_uint(p_new);
                
                uint32_t sign = p_int & 0x80000000;
                uint32_t exp  = p_int & 0x7F800000;
                uint32_t mant = p_int & 0x007FFFFF;
                
                uint32_t mant_mod = (((mant + 1) * 31337) & 0x007FFFFF) & punning_mask;
                
                p_new = __uint_as_float(sign | exp | mant_mod);
                params[idx] = static_cast<scalar_t>(p_new);
            }
            exp_avg[idx] = 0.0f;
            exp_avg_sq[idx] *= 0.1f;
        }
    }
}

#define CHECK_CUDA(x) TORCH_CHECK(x.device().is_cuda(), #x " must be a CUDA tensor")
#define CHECK_CONTIGUOUS(x) TORCH_CHECK(x.is_contiguous(), #x " must be contiguous")
#define CHECK_INPUT(x) CHECK_CUDA(x); CHECK_CONTIGUOUS(x)

void adamv_step_cuda(
    at::Tensor p,
    at::Tensor grad,
    at::Tensor exp_avg,
    at::Tensor exp_avg_sq,
    at::Tensor direcao,
    float lr,
    float beta1,
    float beta2,
    float eps,
    float weight_decay,
    float progresso,
    float bakh_thresh_eff,
    int step,
    int D,
    bool enable_cooling,
    bool enable_brake,
    bool omni_triggered,
    int64_t punning_mask) 
{
    CHECK_INPUT(p);
    CHECK_INPUT(grad);
    CHECK_INPUT(exp_avg);
    CHECK_INPUT(exp_avg_sq);
    CHECK_INPUT(direcao);

    int numel = p.numel();
    int blocks = (numel + BLOCK_SIZE - 1) / BLOCK_SIZE;

    float bias_correction2 = 1.0f - std::pow(static_cast<float>(beta2), static_cast<float>(step));

    cudaStream_t stream = at::cuda::getCurrentCUDAStream();

    AT_DISPATCH_FLOATING_TYPES_AND_HALF(p.scalar_type(), "adamv_prepare", [&] {
        adamv_prepare_kernel<scalar_t><<<blocks, BLOCK_SIZE, 0, stream>>>(
            exp_avg.data_ptr<float>(),
            exp_avg_sq.data_ptr<float>(),
            direcao.data_ptr<float>(),
            grad.data_ptr<scalar_t>(),
            beta1, beta2, bias_correction2, eps, step, numel
        );
    });

    // Compute norm asynchronously on GPU
    at::Tensor norm_tensor = at::linalg_norm(direcao);
    
    float cooling_factor;
    if (progresso < 0.1f) {
        cooling_factor = 0.01f + (progresso / 0.1f) * 0.99f;
    } else {
        float cos_progresso = (progresso - 0.1f) / 0.9f;
        cooling_factor = 1.0f;
    }
    float wd_factor = 0.5f * (1.0f + std::cos(M_PI * progresso));

    AT_DISPATCH_FLOATING_TYPES_AND_HALF(p.scalar_type(), "adamv_update", [&] {
        adamv_update_kernel<scalar_t><<<blocks, BLOCK_SIZE, 0, stream>>>(
            p.data_ptr<scalar_t>(),
            grad.data_ptr<scalar_t>(),
            exp_avg.data_ptr<float>(),
            exp_avg_sq.data_ptr<float>(),
            direcao.data_ptr<float>(),
            norm_tensor.data_ptr<float>(),
            progresso, cooling_factor, bakh_thresh_eff, bias_correction2, eps, wd_factor, lr, weight_decay, numel, D, step,
            enable_cooling, enable_brake,
            omni_triggered, static_cast<uint32_t>(punning_mask)
        );
    });
}

"""

cpp_source = """
#include <torch/extension.h>
#include <ATen/cuda/CUDAContext.h>

void adamv_step_cuda(
    at::Tensor p,
    at::Tensor grad,
    at::Tensor exp_avg,
    at::Tensor exp_avg_sq,
    at::Tensor direcao,
    float lr,
    float beta1,
    float beta2,
    float eps,
    float weight_decay,
    float progresso,
    float bakh_thresh_eff,
    int step,
    int D,
    bool enable_cooling,
    bool enable_brake,
    bool omni_triggered,
    int64_t punning_mask
);
"""

print('Compiling JIT C++ Kernel for Kaggle T4...')
adamv_cuda = load_inline(
    name='adamv_cuda_v8',
    cpp_sources=cpp_source,
    cuda_sources=cuda_source,
    functions=['adamv_step_cuda'],
    with_cuda=True,
    extra_cuda_cflags=['-O3', '-use_fast_math', '-arch=sm_75']
)
print('JIT Compilation complete! Kernel injected globally.')

In [ ]:
import torch
import math
import torch
import math
import torch
import math
import torch
import math
import torch
import math
import torch
import math
import torch
import math
import torch
import math
import torch
import math
import torch
import math
import torch
import math
import torch
import math

import torch
import math

class AdamV(torch.optim.Optimizer):
    """
    AdamV (Adam Vedic) Optimizer - Pure Python Version.
    AdamV 3.1: Harmonic Refactor (In-Place VRAM Opt, OMNI State Fix, Modular Flags)
    """
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, 
                 weight_decay=0.01, total_steps=10000, 
                 bakhshali_threshold=15.0, enable_omni=True,
                 lp_kappa=0.1, lp_omega=10.0, punning_mask=0xFFFFE000,
                 enable_ignition=False, enable_cooling=True, enable_brake=True):
                 
        if not 0.0 <= lr:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay,
                        total_steps=total_steps, bakhshali_threshold=bakhshali_threshold,
                        enable_omni=enable_omni, lp_kappa=lp_kappa, lp_omega=lp_omega, 
                        punning_mask=punning_mask,
                        enable_ignition=enable_ignition, enable_cooling=enable_cooling, enable_brake=enable_brake)
        super(AdamV, self).__init__(params, defaults)
        
        if 'omni_loss_ema' not in self.param_groups[0]:
            self.param_groups[0]['omni_loss_ema'] = float('inf')
            self.param_groups[0]['omni_patience'] = 0.0
            self.param_groups[0]['omni_clock_reset_step'] = 0
            self.param_groups[0]['omni_global_step'] = 0

    @torch.no_grad()
    def step(self, closure=None, current_loss=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
                
        if current_loss is not None:
            loss = current_loss

        g_state = self.param_groups[0]
        g_state['omni_global_step'] += 1
        current_step = g_state['omni_global_step']
        
        omni_triggered = False
        if loss is not None and len(self.param_groups) > 0 and self.param_groups[0]['enable_omni']:
            loss_val = float(loss) if isinstance(loss, torch.Tensor) else loss
            if g_state['omni_loss_ema'] == float('inf'):
                g_state['omni_loss_ema'] = loss_val
                g_state['omni_patience'] = 0.0
            else:
                g_state['omni_loss_ema'] = 0.9 * g_state['omni_loss_ema'] + 0.1 * loss_val
                
            is_worse = loss_val > g_state['omni_loss_ema'] * 0.99
            g_state['omni_patience'] = g_state['omni_patience'] + 1.0 if is_worse else 0.0
            
            patience_limit = max(500, int(self.param_groups[0]['total_steps'] * 0.05))
            if g_state['omni_patience'] >= patience_limit:
                omni_triggered = True
                g_state['omni_patience'] = 0.0
                
        for group in self.param_groups:
            lr_max = group['lr']
            total_steps = group['total_steps']
            
            # Autonomous Ignition
            if group.get('enable_ignition', True):
                ignition = min(1.0, current_step / max(1.0, total_steps * 0.10))
                lr_max = lr_max * ignition
                
            # Diophantine QMC Jitter (Time-Domain Scrambling)
            # Adds a low-discrepancy jitter to the base learning rate
            jitter_scale = 0.15 * max(0.0, 1.0 - (current_step / max(1, total_steps)))
            lr_max = lr_max * (1.0 + jitter_scale * ((current_step * 0.6180339887) % 1.0 - 0.5))

            beta1, beta2 = group['betas']
            eps = group['eps']
            weight_decay = group['weight_decay']
            total_steps = group['total_steps']
            bakh_thresh = group['bakhshali_threshold']
            lp_kappa = group['lp_kappa']
            lp_omega = group['lp_omega']
            punning_mask = group['punning_mask']
            
            internal_step = current_step - g_state['omni_clock_reset_step']
            progresso = min(1.0, internal_step / max(1, total_steps))

            # Resfriamento MacroscÃ³pico (Cosine Decay)
            decay_cosseno = 0.01 + 0.99 * 0.5 * (1.0 + math.cos(math.pi * progresso))
            
            # Log-Periodic Topological Cooling (Nyquist Capped)
            onda = 0.3 * math.cos(math.pi * 4.0 * progresso)
            
            # O envelope total funde o decaimento com a onda
            lr_max = lr_max * decay_cosseno * (1.0 + onda)

            
            LP_Fator = 1.0 + lp_kappa * math.cos(lp_omega * math.log(1.0 + progresso * 10.0))
            bakh_thresh_eff = bakh_thresh * LP_Fator
            
            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p, memory_format=torch.preserve_format, dtype=torch.float32)
                    state['exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format, dtype=torch.float32)
                
                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                state['step'] += 1
                
                # 100% In-Place BRCM
                grad_f = grad.float()
                
                # Update exp_avg_sq first!
                exp_avg_sq.mul_(beta2).addcmul_(grad_f, grad_f, value=1.0 - beta2)
                
                bias_correction2 = 1.0 - beta2 ** state['step']
                v_hat = exp_avg_sq / bias_correction2
                sqrt_v_hat = v_hat.sqrt()
                
                # Excess Shock Isolator (delta)
                # Tighter tuning: 1.0x margin to prevent excess momentum in sharp minima
                delta = torch.clamp(torch.abs(grad_f) - 1.5 * sqrt_v_hat, min=0.0)
                
                denom_brcm = sqrt_v_hat + delta + eps
                bakh_residual = (delta * delta) / (denom_brcm * 2.0)
                
                # Full curvature shift to avoid carrying too much momentum
                curvature_shift = bakh_residual / (sqrt_v_hat + eps)
                beta1_eff = torch.exp(-0.5 * curvature_shift) * beta1
                
                bias_correction1_dynamic = 1.0 - beta1_eff ** state['step']
                
                # Update momentum In-Place
                exp_avg.mul_(beta1_eff).add_(grad_f * (1.0 - beta1_eff))
                
                direcao = (exp_avg / bias_correction1_dynamic) / (sqrt_v_hat + eps)
                
                norm_dir_padrao = torch.linalg.norm(direcao) / math.sqrt(p.numel())
                
                if group.get('enable_cooling', False):
                    envelope = (1.0 + progresso) / (progresso + norm_dir_padrao + eps)
                    if progresso < 0.1:
                        cooling_factor = 0.01 + (progresso / 0.1) * 0.99
                    else:
                        cos_progresso = (progresso - 0.1) / 0.9
                        cooling_factor = 1.0
                    lr_efetivo = lr_max * torch.clamp(envelope * cooling_factor, max=1.5)
                else:
                    lr_efetivo = lr_max
                
                a = direcao.mul_(lr_efetivo)
                
                explosao_mask = torch.abs(grad) > (bakh_thresh_eff * sqrt_v_hat)
                
                # Dimensional Mismatch Fix: parameter absolute magnitude
                # Tightened Tuning: Stronger brake penalty for late-stage stability
                denom = p.abs() + (a.abs() * 2.0) + eps
                correction = (a * a) / (denom * 2.0)
                
                bakhshali_brake = a - (torch.sign(a) * correction)
                
                if group.get('enable_brake', True):
                    step_size = torch.where(explosao_mask, bakhshali_brake, a)
                else:
                    step_size = a
                    
                # Orthogonal Truncated Levy Flight Injection
                u = torch.rand_like(p) * 0.999 + 0.0005
                cauchy = torch.tan(math.pi * (u - 0.5))
                cauchy = torch.clamp(cauchy, -500.0, 500.0)
                levy_scale = sqrt_v_hat / (sqrt_v_hat + 0.1)
                levy_mult = 0.01 * max(0.0, 1.0 - progresso)
                levy_jump = lr_efetivo * levy_mult * cauchy * levy_scale
                step_size.sub_(levy_jump)
                
                if weight_decay != 0:
                    wd_factor = 0.5 * (1.0 + math.cos(math.pi * progresso))
                    p.mul_(1.0 - lr_max * weight_decay * wd_factor)
                    
                p.sub_(step_size)
                
                if omni_triggered:
                    if p.dtype == torch.float32:
                        p_int = p.view(torch.int32)
                        sign_exp = p_int & 0xFF800000
                        mant = p_int & 0x007FFFFF
                        
                        mask_val = int(punning_mask)
                        if mask_val > 0x7FFFFFFF:
                            mask_val -= 0x100000000
                        
                        scrambled_mant = (((mant + 1) * 31337) & 0x007FFFFF) & mask_val
                        p_new = (sign_exp | scrambled_mant).view(torch.float32)
                        p.copy_(p_new)
                    
                    state['exp_avg'].zero_()
                    state['exp_avg_sq'].mul_(0.1)
                    state['step'] = 0

        if omni_triggered:
            g_state['omni_clock_reset_step'] = current_step
                
        return loss

class AdamVCpp(torch.optim.Optimizer):
    """
    AdamV (Adam Vedic) Optimizer - C++ Fused Kernel Version.
    AdamV 3.1: Harmonic Refactor
    """
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, 
                 weight_decay=0.01, total_steps=10000, 
                 bakhshali_threshold=15.0, enable_omni=True,
                 lp_kappa=0.1, lp_omega=10.0, punning_mask=0xFFFFE000,
                 enable_ignition=False, enable_cooling=True, enable_brake=True):
                 
        self.adamv_cpp = None
            
        self.adamv_cuda = adamv_cuda
            
        if not 0.0 <= lr:
            raise ValueError(f"Invalid learning rate: {lr}")
            
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay,
                        total_steps=total_steps, bakhshali_threshold=bakhshali_threshold,
                        enable_omni=enable_omni, lp_kappa=lp_kappa, lp_omega=lp_omega, 
                        punning_mask=punning_mask,
                        enable_ignition=enable_ignition, enable_cooling=enable_cooling, enable_brake=enable_brake)
        super(AdamVCpp, self).__init__(params, defaults)
        
        if 'omni_loss_ema' not in self.param_groups[0]:
            self.param_groups[0]['omni_loss_ema'] = float('inf')
            self.param_groups[0]['omni_patience'] = 0.0
            self.param_groups[0]['omni_clock_reset_step'] = 0
            self.param_groups[0]['omni_global_step'] = 0

    @torch.no_grad()
    def step(self, closure=None, current_loss=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        if current_loss is not None:
            loss = current_loss

        g_state = self.param_groups[0]
        g_state['omni_global_step'] += 1
        current_step = g_state['omni_global_step']
        
        omni_triggered = False
        if loss is not None and len(self.param_groups) > 0 and self.param_groups[0]['enable_omni']:
            loss_val = float(loss) if isinstance(loss, torch.Tensor) else loss
            if g_state['omni_loss_ema'] == float('inf'):
                g_state['omni_loss_ema'] = loss_val
                g_state['omni_patience'] = 0.0
            else:
                g_state['omni_loss_ema'] = 0.9 * g_state['omni_loss_ema'] + 0.1 * loss_val
                
            is_worse = loss_val > g_state['omni_loss_ema'] * 0.99
            g_state['omni_patience'] = g_state['omni_patience'] + 1.0 if is_worse else 0.0
            
            patience_limit = max(500, int(self.param_groups[0]['total_steps'] * 0.05))
            if g_state['omni_patience'] >= patience_limit:
                omni_triggered = True
                g_state['omni_patience'] = 0.0
                
        for group in self.param_groups:
            lr_max = group['lr']
            total_steps = group['total_steps']
            
            if group.get('enable_ignition', True):
                ignition = min(1.0, current_step / max(1.0, total_steps * 0.10))
                lr_max = lr_max * ignition
                
            # Diophantine QMC Jitter (Time-Domain Scrambling)
            jitter_scale = 0.15 * max(0.0, 1.0 - (current_step / max(1, total_steps)))
            lr_max = lr_max * (1.0 + jitter_scale * ((current_step * 0.6180339887) % 1.0 - 0.5))

            beta1, beta2 = group['betas']
            eps = group['eps']
            weight_decay = group['weight_decay']
            total_steps = group['total_steps']
            bakh_thresh = group['bakhshali_threshold']
            lp_kappa = group['lp_kappa']
            lp_omega = group['lp_omega']
            punning_mask = group['punning_mask']
            
            internal_step = current_step - g_state['omni_clock_reset_step']
            progresso = min(1.0, internal_step / max(1, total_steps))

            # Resfriamento MacroscÃ³pico (Cosine Decay)
            decay_cosseno = 0.01 + 0.99 * 0.5 * (1.0 + math.cos(math.pi * progresso))
            
            # Log-Periodic Topological Cooling (Nyquist Capped)
            onda = 0.3 * math.cos(math.pi * 4.0 * progresso)
            
            # O envelope total funde o decaimento com a onda
            lr_max = lr_max * decay_cosseno * (1.0 + onda)
            
            LP_Fator = 1.0 + lp_kappa * math.cos(lp_omega * math.log(1.0 + progresso * 10.0))
            bakh_thresh_eff = bakh_thresh * LP_Fator
            
            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p, memory_format=torch.preserve_format, dtype=torch.float32)
                    state['exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format, dtype=torch.float32)
                    state['direcao_buffer'] = torch.empty_like(p, memory_format=torch.preserve_format, dtype=torch.float32)
                
                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                state['step'] += 1
                
                # C++ Fused Kernel Call
                # OMNI logic is pushed to the END inside the C++ Kernel now
                mask_val = int(punning_mask)
                if mask_val > 0x7FFFFFFF:
                    mask_val -= 0x100000000
                    
                if p.is_cpu:
                    self.adamv_cpp.adamv_step_cpu(
                        p, grad, exp_avg, exp_avg_sq, state['direcao_buffer'],
                        lr_max, beta1, beta2, eps, weight_decay,
                        float(progresso), float(bakh_thresh_eff), state['step'], p.numel(),
                        bool(group.get('enable_cooling', False)), bool(group.get('enable_brake', True))
                    )
                    if omni_triggered:
                        if p.dtype == torch.float32:
                            p_int = p.view(torch.int32)
                            sign_exp = p_int & 0xFF800000
                            mant = p_int & 0x007FFFFF
                            scrambled_mant = (((mant + 1) * 31337) & 0x007FFFFF) & mask_val
                            p_new = (sign_exp | scrambled_mant).view(torch.float32)
                            p.copy_(p_new)
                        state['exp_avg'].zero_()
                        state['exp_avg_sq'].mul_(0.1)
                        state['step'] = 0
                elif p.is_cuda and self.adamv_cuda is not None and hasattr(self.adamv_cuda, 'adamv_step_cuda'):
                    self.adamv_cuda.adamv_step_cuda(
                        p, grad, exp_avg, exp_avg_sq, state['direcao_buffer'],
                        lr_max, beta1, beta2, eps, weight_decay,
                        float(progresso), float(bakh_thresh_eff), state['step'], p.numel(),
                        bool(group.get('enable_cooling', False)), bool(group.get('enable_brake', True)),
                        bool(omni_triggered), mask_val
                    )
                    if omni_triggered:
                        state['step'] = 0
                else:
                    # Python fallback para GPU - 100% In-Place BRCM
                    grad_f = grad.float()
                    
                    # Update exp_avg_sq first!
                    exp_avg_sq.mul_(beta2).addcmul_(grad_f, grad_f, value=1.0 - beta2)
                    
                    bias_correction2 = 1.0 - beta2 ** state['step']
                    v_hat = exp_avg_sq / bias_correction2
                    sqrt_v_hat = v_hat.sqrt()
                    
                    # Excess Shock Isolator (delta)
                    # Tighter tuning: 1.0x margin to prevent excess momentum in sharp minima
                    delta = torch.clamp(torch.abs(grad_f) - 1.5 * sqrt_v_hat, min=0.0)
                    
                    denom_brcm = sqrt_v_hat + delta + eps
                    bakh_residual = (delta * delta) / (denom_brcm * 2.0)
                    
                    # Full curvature shift to avoid carrying too much momentum
                    curvature_shift = bakh_residual / (sqrt_v_hat + eps)
                    beta1_eff = torch.exp(-0.5 * curvature_shift) * beta1
                    
                    bias_correction1_dynamic = 1.0 - beta1_eff ** state['step']
                    
                    # Update momentum In-Place
                    exp_avg.mul_(beta1_eff).add_(grad_f * (1.0 - beta1_eff))
                    
                    direcao = (exp_avg / bias_correction1_dynamic) / (sqrt_v_hat + eps)
                    
                    norm_dir = torch.linalg.norm(direcao) / math.sqrt(p.numel())
                    if group.get('enable_cooling', False):
                        envelope = (1.0 + progresso) / (progresso + norm_dir + eps)
                        if progresso < 0.1:
                            cooling = 0.01 + (progresso / 0.1) * 0.99
                        else:
                            cos_progresso = (progresso - 0.1) / 0.9
                            cooling = 0.01 + 0.5 * 0.99 * (1.0 + math.cos(math.pi * cos_progresso))
                        lr_efetivo = lr_max * torch.clamp(envelope * cooling, max=1.5)
                    else:
                        lr_efetivo = lr_max
                    
                    a = direcao.mul_(lr_efetivo)
                    sqrt_v_hat = v_hat.sqrt()
                    
                    explosao_mask = torch.abs(grad) > (bakh_thresh_eff * sqrt_v_hat)
                    
                    # Dimensional Mismatch Fix: parameter absolute magnitude
                    # Tightened Tuning: Stronger brake penalty for late-stage stability
                    denom = p.abs() + (a.abs() * 2.0) + eps
                    correction = (a * a) / (denom * 2.0)
                    bakhshali_brake = a - (torch.sign(a) * correction)
                    
                    if group.get('enable_brake', True):
                        step_size = torch.where(explosao_mask, bakhshali_brake, a)
                    else:
                        step_size = a
                        
                    # Orthogonal Truncated Levy Flight Injection
                    u = torch.rand_like(p) * 0.999 + 0.0005
                    cauchy = torch.tan(math.pi * (u - 0.5))
                    cauchy = torch.clamp(cauchy, -500.0, 500.0)
                    levy_scale = sqrt_v_hat / (sqrt_v_hat + 0.1)
                    levy_mult = 0.01 * max(0.0, 1.0 - progresso)
                    levy_jump = lr_efetivo * levy_mult * cauchy * levy_scale
                    step_size.sub_(levy_jump)
                        
                    if weight_decay != 0:
                        wd_factor = 0.5 * (1.0 + math.cos(math.pi * progresso))
                        p.mul_(1.0 - lr_max * weight_decay * wd_factor)
                        
                    p.sub_(step_size)
                    
                    if omni_triggered:
                        if p.dtype == torch.float32:
                            p_int = p.view(torch.int32)
                            sign_exp = p_int & 0xFF800000
                            mant = p_int & 0x007FFFFF
                            
                            scrambled_mant = (((mant + 1) * 31337) & 0x007FFFFF) & mask_val
                            p_new = (sign_exp | scrambled_mant).view(torch.float32)
                            p.copy_(p_new)
                        
                        state['exp_avg'].zero_()
                        state['exp_avg_sq'].mul_(0.1)
                        state['step'] = 0

        if omni_triggered:
            g_state['omni_clock_reset_step'] = current_step
                        
        return loss


In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt


# =============================================================================
# 1. Strict Determinism
# =============================================================================
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def get_deterministic_generator(seed: int):
    g = torch.Generator()
    g.manual_seed(seed)
    return g

# =============================================================================
# 2. Vision Pipeline (CIFAR-100)
# =============================================================================
def get_vision_dataloaders(batch_size=128, seed=42):
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5071, 0.4865, 0.4409], 
                             std=[0.2673, 0.2564, 0.2762])
    ])
    
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5071, 0.4865, 0.4409], 
                             std=[0.2673, 0.2564, 0.2762])
    ])
    
    # Using CIFAR-100 to push complexity further than CIFAR-10
    trainset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform_train)
    testset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform_test)
    
    gen = get_deterministic_generator(seed)
    
    train_dl = DataLoader(trainset, batch_size=batch_size, shuffle=True, 
                          num_workers=2, worker_init_fn=seed_worker, generator=gen, drop_last=True)
    val_dl = DataLoader(testset, batch_size=batch_size, shuffle=False)
    
    return train_dl, val_dl

# =============================================================================
# 3. Model Architecture (Modified ResNet-18 for 32x32)
# =============================================================================
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out

class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=100):
        super(ResNet, self).__init__()
        self.in_planes = 64

        # Modified Stem for 32x32 inputs (CIFAR)
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.linear = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out

def ResNet18():
    return ResNet(BasicBlock, [2, 2, 2, 2])

# =============================================================================
# 4. Decoupled Weight Decay & Training Loop
# =============================================================================
def configure_optimizers(model, weight_decay, learning_rate, is_adamv=False):
    # Decouple weight decay: disable for BatchNorm and biases
    decay_params = []
    nodecay_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if len(param.shape) == 1 or name.endswith(".bias"):
            nodecay_params.append(param)
        else:
            decay_params.append(param)
            
    optim_groups = [
        {'params': decay_params, 'weight_decay': weight_decay},
        {'params': nodecay_params, 'weight_decay': 0.0}
    ]
    
    if is_adamv:
        return AdamVCpp(optim_groups, lr=learning_rate, betas=(0.9, 0.999), enable_omni=True)
    else:
        return torch.optim.AdamW(optim_groups, lr=learning_rate, betas=(0.9, 0.999), eps=1e-8)

@torch.no_grad()
def evaluate(model, val_dl, device):
    model.eval()
    correct_1 = 0
    total = 0
    losses = []
    
    for inputs, targets in val_dl:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        loss = F.cross_entropy(outputs, targets)
        losses.append(loss.item())
        
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct_1 += predicted.eq(targets).sum().item()
        
    model.train()
    acc = 100.0 * correct_1 / total
    return sum(losses)/len(losses), acc

def run_vision_arena():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    epochs = 20
    seeds = [42, 1337] # Multi-seed rigor
    
    results = {"AdamW": [], "AdamVCpp": []}
    
    for seed in seeds:
        print(f"\n--- Running Seed {seed} ---")
        seed_everything(seed)
        train_dl, val_dl = get_vision_dataloaders(batch_size=128, seed=seed)
        
        for opt_name in ["AdamW", "AdamVCpp"]:
            print(f"Training with {opt_name}...")
            seed_everything(seed)
            model = ResNet18().to(device)
            
            lr = 1e-3 if opt_name == "AdamW" else 3e-3
            wd = 0.05
            optimizer = configure_optimizers(model, wd, lr, is_adamv=(opt_name=="AdamVCpp"))
            
            # Scheduler ONLY for AdamW (AdamV usa nativo)
            if opt_name == "AdamW":
                scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, steps_per_epoch=len(train_dl), epochs=epochs, pct_start=0.1)
                
            history_acc = []
            
            for epoch in range(epochs):
                model.train()
                for batch_idx, (inputs, targets) in enumerate(train_dl):
                    inputs, targets = inputs.to(device), targets.to(device)
                    optimizer.zero_grad(set_to_none=True)
                    outputs = model(inputs)
                    loss = F.cross_entropy(outputs, targets, label_smoothing=0.1)
                    loss.backward()
                    
                    optimizer.step()
                    if opt_name == "AdamW":
                        scheduler.step()
                        
                val_loss, val_acc = evaluate(model, val_dl, device)
                print(f"Epoch {epoch}: Val Acc {val_acc:.2f}%")
                history_acc.append((epoch, val_acc))
                
            results[opt_name].append(history_acc)
            
    # Plotting Logic
    plt.figure(figsize=(10, 6))
    for opt_name, histories in results.items():
        epochs_arr = [h[0] for h in histories[0]]
        accs = np.array([[h[1] for h in history] for history in histories])
        mean_acc = accs.mean(axis=0)
        std_acc = accs.std(axis=0)
        plt.plot(epochs_arr, mean_acc, label=opt_name, marker='o')
        plt.fill_between(epochs_arr, mean_acc - std_acc, mean_acc + std_acc, alpha=0.2)
        
    plt.title("Vision Arena (ResNet-18 on CIFAR-100)")
    plt.xlabel("Epochs")
    plt.ylabel("Validation Accuracy (%)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig("assets/vision_arena.png")
    print("Vision Arena Benchmark complete! Saved to assets/vision_arena.png")

if __name__ == "__main__":
    run_vision_arena()

